# Data Visualisation

In this phase we will analyse in details the singular features of each dataset, using graphs in order to get a better grasp of their contents, before formulating hypothesis and diving deeper into the correlations between them.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import functions as fun
import numpy as np
import squarify

In [ ]:
actors_df = pd.read_csv('../../Data/DataFinal/actors_v2.csv')
countries_df = pd.read_csv('../../Data/DataFinal/countries_v2.csv')
crew_df = pd.read_csv('../../Data/DataFinal/crew_v2.csv')
genres_df = pd.read_csv('../../Data/DataFinal/genres_v2.csv')
languages_df = pd.read_csv('../../Data/DataFinal/languages_v2.csv')
movies_df = pd.read_csv('../../Data/DataFinal/movies_v2.csv', parse_dates=['date'], low_memory=False)
releases_df = pd.read_csv('../../Data/DataFinal/releases_v2.csv', parse_dates=['date'])
reviews_df = pd.read_csv('../../Data/DataFinal/rotten_tomatoes_reviews_v2.csv', parse_dates=['review_date'])
studios_df = pd.read_csv('../../Data/DataFinal/studios_v2.csv')
oscar_df = pd.read_csv('../../Data/DataFinal/the_oscar_awards_v2.csv', parse_dates=['year_ceremony'])

In [ ]:
plt.style.use('ggplot')

## Actors Study

### Top 15 Most Proficient Actors by Number of Film


In [ ]:
actor_count = actors_df.groupby('name')['id'].nunique().reset_index()
# Sorting the actors by the number of films they have appeared in
actor_count = actor_count.sort_values(by='id', ascending=False)

# Bar plot
plt.figure(figsize=(12, 6))
sns.barplot(x='id', y='name', data=actor_count.head(15), hue='name', palette='viridis')
plt.xlabel('Number of Films')
plt.ylabel('Actor')
plt.title('Top 15 Most Proficient Actors by Number of Films')
#fig.update_layout(margin={"r":10,"t":40,"l":10,"b":40})
plt.show()

We can immediately see how **Mel Blanc** is easily the most prolific actor in the dataset. He was an American voice actor, who voiced numerous characters from extremely famous films and cartoons. This implies that it is much more common for voice actors to take part to many films, compared to actors, whose work dedicated to each film probably takes longer on average.

**Bess Flowers** specialised in working as an extra, which explains how she participated in over 700 films during her career.

**Jagathy Sreekumar, Mammootty and Mohanlal** were Indian actors, and their extensive careers are due to the high prolificacy of the Indian cinema industry.

### Top 15 Actors Who Played the Most Different Roles

In [ ]:
actor_role_counts = actors_df.groupby('name')['role'].nunique()

# Select the top 15 actors who have played the most different roles
top_actors_roles = actor_role_counts.nlargest(15).reset_index()
top_actors_roles.columns = ['Actor', 'Unique Roles']

# Create Treemap with Plotly Express
fig = px.treemap(top_actors_roles,
                 path=['Actor'],
                 values='Unique Roles',
                 color='Unique Roles',
                 color_continuous_scale='magma',
                 title='Top 15 Actors Who Played the Most Different Roles')
fig.update_traces(hovertemplate="<b>%{label}</b><br>Unique Roles: %{value}")
fig.update_layout(margin={"r":10,"t":40,"l":10,"b":40})
fig.show()

Not only **Mel Blanc** is the actor who played in more films, but he is also the actor with the biggest number of unique roles, which is reasonable considering his outstanding career.

It's interesting to notice how the top 3 changed, even tho **Bess Flowers** was second regarding the number of
roles played, here she is surpassed by **Eric Roberts**, probably because she often played the same character across multiple films. Considering how common it is for a film to have sequels, this is the case for many other actors as well: in this chart we don't see some of them that were instead present in the previous graph, since they played the same role multiple times.

## Country Trends


### Top 15 Countries Who Hosted the Most Films

In [ ]:
# Count number of film per country
countries_count = countries_df['country'].value_counts().head(15)

# Create the bar chart
plt.figure(figsize=(10, 6))
countries_count.plot(kind='bar', color=fun.color_palette(countries_count), edgecolor='black')

# Add labels and title
plt.title('Number of Film Recorded per Country', fontsize=16)
plt.xlabel('Country', fontsize=14)
plt.ylabel('Number of Film', fontsize=14)
plt.xticks(rotation=45)

# Show the plot
plt.show()

We can see that the **USA** is the Country that produced the biggest number of films recorded in our dataset, with more than three times the amount of films compared to France, in second position.

The bar chart underlines this interesting fact: there is a huge gap between the first Country at the top and the rest. At the same time though, we can see that, apart from the USA, the other countries have a pretty constant decrease in the number of films recorded:
* top 2 - 5: is constant on 45.000 films;
* top 6 - 7: Canada and India have almost the same numbers of films;
* top 8 - 15: There is a stable decline in the number of films per country.

**France, the United Kingdom, Germany** and **Japan** are all very prolific Countries. Italy is too, even if it produced about half the films of these 4.

## Crew Trends

### Top 15 Directors with the Most Films

In [ ]:
fun.top15_director(crew_df)

**Dave Fleischer** is the person who directed most of the films in our dataframe, with well over 500 films to his name.

We can also notice that the majority of Directors displayed in this graph are Americans: this is no coincidence since, as we saw earlier, the USA is the Country that made the most films.

## Genre Trends

### The 15 Most Common Genres

In [ ]:
# Count the frequency of any genre
genre_counts = genres_df['genre'].value_counts().head(15)
colors = sns.color_palette('viridis', len(genre_counts))

# Donut chart
plt.figure(figsize=(7, 10))
wedges, texts, autotexts = plt.pie(genre_counts.values, labels=genre_counts.index, colors=colors, autopct='%1.1f%%', startangle=140,
    pctdistance=0.85, textprops={'fontsize': 10, 'weight': 'bold'}
)
centre_circle = plt.Circle((0, 0), 0.60, color='white')
plt.gca().add_artist(centre_circle)

plt.title('Top 15 Most Common Genres', fontsize=16, weight='bold', pad=20)
plt.tight_layout()
plt.show()

**Drama** is the most popular genre in our dataset, being the genre of almost a quarter of the total number of films.

We can see that Drama, along with the two genres that follow it, **Documentaries** and **Comedies**, is significantly more popular than the others, accounting for more than half of the films we will be analysing.

## Movie Trends

### Film Lengths

In [ ]:
movies_temp_df = movies_df.dropna(subset=['minute'])
max_duration = movies_temp_df['minute'].max()
bins = np.arange(0, max_duration + 1, 10)

fig, ax = plt.subplots(figsize=(10, 6))

sns.histplot(movies_temp_df['minute'], bins=bins, kde=False, color='green', ax=ax)
ax.set_title('Histogram of Film Durations')
ax.set_xlabel('Duration (minutes)')
ax.set_ylabel('Number of Films')
ax.set_xlim(0, bins[-1])
ax.set_xticks(np.arange(0, bins[-1]+1, 50))

plt.show()

This graph shows us the different lengths of films in 10 minutes intervals. We can immediately notice how the vast majority of them has a duration included **between 0 and 200 minutes**, with a substantial amount of outlier values that reach almost 400 minutes.

The most common duration is between 0 and 10 minutes, followed by the 90-100 and 10-20 ranges. This tells us that there is an abundance of **short films**, as well as films with a duration on an hour and a half.

### Film Ratings Before and After 2000


In [ ]:
fun.movie_rating_century(movies_df)

Looking at them side by side, there isn't a significant change between the 2 charts. The majority of films directed _before_ 2000 have a rating of **3**, exactly as the films created _after_ 2000.

The most noticeable difference is the number of films with a rating of 2. Even though we could think that the films made _after_ 2000 would be made with a more advanced technology, almost 4.000 of them have a low rating of **2**, while only 700 films made _before_ 2000 scored that.

Another thing to notice is that, despite the fact that the films directed _before_ 2000 are less in number than the films created _after_, both categories have almost the same number of films with a rating of **5**, which is in both cases around barely 20 movies.

## Release Trends


### Most Common Film Release Types


In [ ]:
release_counts = releases_df['type'].value_counts()

# Pie chart
plt.figure(figsize=(6, 6))
plt.pie(release_counts, labels=release_counts.index, autopct='%1.1f%%', startangle=140, colors=fun.color_palette(release_counts), textprops={'fontsize': 14, 'weight': 'bold'} )
plt.title('Distribution of Film Release Types', fontsize=14)

plt.subplots_adjust(top=1.0)
plt.axis('equal')
plt.show()

### Top 15 Most Common Genre Released in Theatrical


In [ ]:
fun.plot_most_common_genre_theatrical(releases_df, genres_df)

The most common type of release is **Theatrical**. There is a huge gap between the first position - Theatrical - and the second - Premiere.

It's interesting to notice that, despite the digital transformation in media consumption, traditional theatrical releases still represent the majority of film distribution. Moreover, earlier we saw that **Drama** was the most common genre in our dataframe, and this bar plot, made taking into consideration only the films with a Theatrical release, further confirms it.

## Studio Trends

### Studios that Produced the Most Films


In [ ]:
fun.top_studios_lollipop_chart(studios_df)

In this first chart, we can see that the **BBC** (British Broadcast Company) is the studio that created most of the films in our dataframe. It's immediately followed by **Warner Bros Pictures** and **Columbia Pictures**. The BBC is a British studio, while the other two and most studios in the top portion are Americans.

 If we look at the first position only, having a British studio at the top can be a bit surprising, since we saw in our previous chart that the USA was the Country that produced the most films in our dataset. But if we look at the other studios we notice that most of them are Americans, and there isn't a big gap in numbers between the first position and the rest of the top section, which means our studies still coincide:. if in our data there are a lot more American studios than other Country's, then it makes sense for the USA  to be the Country that produced the most films overall.


### Correlation between BBC Production Count and Film Ratings

In [ ]:
fun.bbc_rating_trend(studios_df, movies_df)

In this chart we analyse better _the number of films produced by the BBC per decade and their average rating_.

Before we look into the details of the chart, we want to clarify that the Ratings scale could deceive: the maximum number these data can reach is 5, but the scale goes up to 10 to allow us to display better the scale of the histogram plot, the Number of Films per decade.

 As we can see, the **rating** of BBC films has been **overall steady** during the decades, always being around 3.5, while the **number of films** created had a **steady increase** over the last century, and **two significant increases** during the 2000s and 2010s. This chart provides a good visualisation of how the number of films doesn't have any effect on the ratings. Another thing we can notice is the big decrease in numbers that can be seen in the 2020s, even if there is nothing atypical in it, as we still have five years left in them.

## Duration Study

### Evolution of Average Ratings and Average Duration:

In [ ]:
avg_mean_df = fun.reviews_movies_merge(movies_df, reviews_df)

# Dual-axis plot
fig_dual = make_subplots(specs=[[{"secondary_y": True}]])

fig_dual.add_trace(
    go.Scatter(x=avg_mean_df['date'], y=avg_mean_df['normalized_score'],
               mode='lines+markers', name='Average Review Score', line=dict(color='blue')),
    secondary_y=False
)
fig_dual.add_trace(
    go.Scatter(x=avg_mean_df['date'], y=avg_mean_df['minute'],
               mode='lines+markers', name='Average Duration (min)', line=dict(color='red')),
    secondary_y=True
)
fig_dual.update_layout(
    title="Evolution of Average Review Score and Duration Over Time",
    xaxis_title="Year",
    width=1100, height=600
)
fig_dual.update_yaxes(title_text="Average Review Score", secondary_y=False)
fig_dual.update_yaxes(title_text="Average Duration (minutes)", secondary_y=True)
fig_dual.show()

From these two graphs, we can analyse the **changes in duration** of films over time, as well as the **fluctuations in review scores** the films that came out in those years received. Straight away we can notice a big difference between the left and right part of the graph.

As we could imagine, **at the beginning of cinematography**, films tended to be much shorter due to technical difficulties and limited techniques. On the contrary, there isn't a clear pattern in review scores, which fluctuate from a maximum of 7.8 to a minimum of 4.8 in just a few years. This could be due to the limited amount of reviews the films made in those years received on Rotten Tomatoes, but it could also mean that the public isn't necessarily used to watching movies so distant from the modern taste, and this leads to it not having a common opinion.

Instead, on the right, we can see that, **over the last 50 years**, the average length of movies stayed relatively the same, while the average review scores slightly decreased and ended up settling around a median value of 6/6.5 each year.
This can be very useful for our research, as it not only shows that the majority of films is considered average by reviewers, neither excellent nor terrible, but also that, despite the rapid technological advancement of the last decades, there isn't a tendency to make loger movies as one could think, but the average duration settled around what we can suppose is considered the best length to tell a story and make it enjoyable for the vast public.